# 05 – Tableau export

Prepares two clean tables for the Tableau dashboard: one row per summer day (`tableau_daily.csv`) and one row per summer (`tableau_yearly.csv`).

In [1]:
import pandas as pd
from pathlib import Path

clean_folder = Path("../data/clean")
tableau_folder = Path("../tableau")
tableau_folder.mkdir(parents=True, exist_ok=True)

daily = pd.read_csv(clean_folder / "daily_summer.csv", parse_dates=["date"])

summer = daily[(daily["in_summer"] == 1) & daily["ems_missions"].notna()].copy()
print(summer.shape)

(822, 13)


In [2]:
# EMS demand index: missions relative to that summer's average (removes the yearly growth)
summer["ems_index"] = (summer["ems_missions"] / summer.groupby("year")["ems_missions"].transform("mean") * 100).round(1)

# Temperature bands + a number to sort them correctly in Tableau
band_labels = ["<20", "20–22", "22–24", "24–26", "26–28", "28–30", "30–32", "32–34", "34+"]
summer["temp_band"] = pd.cut(summer["tmax"], bins=[0, 20, 22, 24, 26, 28, 30, 32, 34, 45], right=False, labels=band_labels)
summer["temp_band_order"] = summer["temp_band"].cat.codes + 1
summer["temp_band"] = summer["temp_band"].astype(str)

# Readable labels for filters and colours
summer["day_label"] = summer["hot_day"].map({1: "Hot day (≥30°C)", 0: "Normal day"})

summer["day_type"] = "Normal day"
summer.loc[(summer["hot_day"] == 1) & (summer["heat_warning"] == 0), "day_type"] = "Hot, no warning"
summer.loc[(summer["hot_day"] == 1) & (summer["heat_warning"] == 1), "day_type"] = "Hot + warning"
summer.loc[(summer["hot_day"] == 0) & (summer["heat_warning"] == 1), "day_type"] = "Warning, not hot"

summer["warning_label"] = summer["warning_level"].map({0: "No warning", 1: "Strong heat stress", 3: "Extreme heat stress"})

# Mission counts as whole numbers
summer["ems_missions"] = summer["ems_missions"].astype(int)
summer["cpr_alert_missions"] = summer["cpr_alert_missions"].astype(int)

print(summer["day_type"].value_counts())

day_type
Normal day          660
Hot + warning        83
Hot, no warning      72
Warning, not hot      7
Name: count, dtype: int64


In [3]:
tableau_daily = summer[["date", "year", "month", "weekday", "weekday_num",
                        "tmax", "temp_band", "temp_band_order",
                        "hot_day", "day_label", "heat_warning", "warning_level", "warning_label", "day_type",
                        "ems_missions", "ems_index", "cpr_alert_missions"]].reset_index(drop=True)

print(tableau_daily.shape)
tableau_daily.head()

(822, 17)


,date,year,month,weekday,weekday_num,tmax,temp_band,temp_band_order,hot_day,day_label,heat_warning,warning_level,warning_label,day_type,ems_missions,ems_index,cpr_alert_missions
0,2018-06-01,2018,6,Friday,4,32.2,32–34,8,1,Hot day (≥30°C),1,1,Strong heat stress,Hot + warning,1357,116.2,20
1,2018-06-02,2018,6,Saturday,5,26.4,26–28,5,0,Normal day,0,0,No warning,Normal day,1188,101.7,11
2,2018-06-03,2018,6,Sunday,6,24.1,24–26,4,0,Normal day,0,0,No warning,Normal day,1194,102.2,14
3,2018-06-04,2018,6,Monday,0,25.4,24–26,4,0,Normal day,0,0,No warning,Normal day,1275,109.1,15
4,2018-06-05,2018,6,Tuesday,1,23.5,22–24,3,0,Normal day,0,0,No warning,Normal day,1171,100.2,18


In [4]:
yearly = summer.groupby("year").agg(
    avg_ems_missions=("ems_missions", "mean"),
    avg_tmax=("tmax", "mean"),
    hot_days=("hot_day", "sum"),
    warning_days=("heat_warning", "sum"),
    hot_days_no_warning=("day_type", lambda s: (s == "Hot, no warning").sum())
)

idx = summer.groupby(["year", "hot_day"])["ems_index"].mean().unstack()
yearly["index_normal_days"] = idx[0]
yearly["index_hot_days"] = idx[1]
yearly["hot_day_effect"] = yearly["index_hot_days"] - yearly["index_normal_days"]

tableau_yearly = yearly.round(1).reset_index()
tableau_yearly

,year,avg_ems_missions,avg_tmax,hot_days,warning_days,hot_days_no_warning,index_normal_days,index_hot_days,hot_day_effect
0,2018,1168.2,27.2,25,17,9,97.9,105.7,7.8
1,2019,1198.5,27.3,25,16,11,97.8,105.9,8.1
2,2020,1183.8,25.9,16,8,8,98.6,106.6,8.0
3,2021,1283.3,25.4,13,5,8,99.1,105.3,6.2
4,2022,1333.3,27.0,21,17,7,99.0,103.4,4.3
5,2023,1259.6,25.3,11,7,5,99.1,106.8,7.8
6,2024,1325.4,25.8,14,5,9,99.1,105.0,5.9
7,2025,1371.6,25.1,11,5,6,99.4,104.2,4.7
8,2026,1429.7,26.2,19,10,9,98.2,106.4,8.2


In [5]:
assert len(tableau_daily) == 822 and tableau_daily["date"].is_unique
assert tableau_daily[["tmax", "ems_missions", "ems_index"]].notna().all().all()
assert len(tableau_yearly) == 9
assert tableau_daily["hot_day"].sum() == 155

print(tableau_daily.groupby("day_label")["ems_index"].mean().round(1))

tableau_daily.to_csv(tableau_folder / "tableau_daily.csv", index=False)
tableau_yearly.to_csv(tableau_folder / "tableau_yearly.csv", index=False)
print("Saved.")

day_label
Hot day (≥30°C)    105.5
Normal day          98.7
Name: ems_index, dtype: float64
Saved.
